In [11]:
#| default_exp lawa

In [12]:
#| hide
import nbdev; nbdev.nbdev_export()

/usr/local/lib/python3.12/dist-packages/nbdev/export.py:80: UserWarning: Notebook '/workspaces/gpt/rugptxl_converter.ipynb' uses `#|export` without `#|default_exp` cell.
Note nbdev2 no longer supports nbdev1 syntax. Run `nbdev_migrate` to upgrade.
See https://nbdev.fast.ai/getting_started.html for more information.
  warn(f"Notebook '{nbname}' uses `#|export` without `#|default_exp` cell.\n"


In [13]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "3"
#os.environ['VLLM_LOGGING_LEVEL'] = 'DEBUG'

In [14]:
#| export
from os import getenv
from vllm import LLM, SamplingParams
from transformers import AutoTokenizer
from front.common import process_seq

model_path = getenv("MODEL")

In [15]:
model_path = 'large/poetry'
model_path = 'large/pelevin'

# loss 1.43 for llama 3.2 1B
model_path = 'lawa'

model_path = 'xl/pelevin'

In [16]:
#| export
full_path = f'./models/{model_path}'
tokenizer = AutoTokenizer.from_pretrained(full_path)
model = LLM(model=full_path, dtype="bfloat16", device="cuda", gpu_memory_utilization=0.40)

INFO 09-30 05:39:58 config.py:1652] Downcasting torch.float32 to torch.bfloat16.
INFO 09-30 05:39:58 llm_engine.py:226] Initializing an LLM engine (v0.6.1.dev238+ge2c6e0a82) with config: model='./models/xl/pelevin', speculative_config=None, tokenizer='./models/xl/pelevin', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config=None, rope_scaling=None, rope_theta=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=2048, download_dir=None, load_format=LoadFormat.AUTO, tensor_parallel_size=1, pipeline_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=auto, quantization_param_path=None, device_config=cuda, decoding_config=DecodingConfig(guided_decoding_backend='outlines'), observability_config=ObservabilityConfig(otlp_traces_endpoint=None, collect_model_forward_time=False, collect_model_execute_time=False), seed=0, served_model_name=./models/xl/pelevin, use_v2_block_m

Loading safetensors checkpoint shards:   0% Completed | 0/2 [00:00<?, ?it/s]


In [7]:
#| export
def iftoken(tokenizer, tokens):
    # returns token id if the given string is one token
    token_ids = [tokenizer.encode(token, add_special_tokens=False) for token in tokens]
    return [id for sublist in token_ids for id in sublist if len(sublist) == 1]

stop_token_ids = iftoken(tokenizer, ['<|endoftext|>','|eot_id|','<|end_of_text|>','<|eot_id|>','<pad>'])

def create_token_blocker(tokenizer, blocked_tokens):
    blocked_token_ids = iftoken(tokenizer, blocked_tokens)
    
    def token_blocker(input_ids, scores):
        if scores.dim() == 2:
            scores[:, list(blocked_token_ids)] = -float('inf')
        elif scores.dim() == 1:
            scores[list(blocked_token_ids)] = -float('inf')
        else:
            raise ValueError(f"Unexpected score tensor shape: {scores.shape}")
        return scores
    
    return token_blocker
    
def get_sampling_params(tokenizer, length: int, num_samples: int, allow_linebreak: bool, temperature: float):
    blocked_tokens = ['[', '(', '\xa0', '*', '­', '~', '_', '\\', '\uf04a', '\ufeff', '\u2028']
    if not allow_linebreak:
        blocked_tokens.extend(['\n', '\n\n',' \n'])
    
    token_blocker = create_token_blocker(tokenizer, blocked_tokens)

    return SamplingParams(
        temperature=temperature,
        max_tokens=length,
        n=num_samples,
        top_p=0.9,
        top_k=-1,
        stop_token_ids=stop_token_ids,
        ignore_eos=True,
        logits_processors=[token_blocker],
        repetition_penalty=2.,
    )

In [8]:
stop_token_ids

[1]

In [9]:
#| export
def get_sample(prompt: str, length: int, num_samples: int, allow_linebreak: bool, temperature: float = 1.0):
    sampling_params = get_sampling_params(tokenizer, length, num_samples, allow_linebreak, temperature)
    outputs = model.generate(prompt, sampling_params)
    
    generated_sequences = [oo.text for o in outputs for oo in o.outputs]
    return process_seq(generated_sequences)


In [10]:
%%time
get_sample('На словах ты Лев Толстой, а на деле', 50, 4, False)

Processed prompts: 100%|██████████| 1/1 [00:00<00:00,  2.82it/s, est. speed input: 25.44 toks/s, output: 565.32 toks/s]

CPU times: user 333 ms, sys: 28.1 ms, total: 361 ms
Wall time: 359 ms


[' хрен собачий», - раздался голос из черной дыры. «Да не это главное в жизни человека!',
 ' непонятно кто. Я бы тебе не дала». Толкнул Женьку кулаком в бок – та даже подпрыгнула от неожиданности: «А что я делаю?',
 ' — простой партийный клоун». Эти слова Орлова звучали из его уст часто и много раз.',
 ' — как Семен Гейченко… Все тебя знают. Ты же сам мне говорил про свой роман с высоты престола…» Ей нравилось это «наполовину». Сейчас она покажет Льву Толстому его место!']